In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import random
import heapq

np.random.seed(42)
random.seed(42)

# ==============================
# CONFIGURATION
# ==============================

NUM_RESTAURANTS = 50
DAYS = 7
TIME_INTERVAL_MIN = 5
START_DATE = datetime(2024, 1, 1)

# ==============================
# STEP 1: GENERATE RESTAURANTS
# ==============================

def generate_restaurants(n):

    cuisines = ["Indian", "Chinese", "Fast Food", "Italian", "Desserts"]
    restaurant_types = ["Cloud Kitchen", "Dine-in", "Quick Service"]

    restaurants = []

    for i in range(n):
        restaurants.append({
            "restaurant_id": i,
            "cuisine": random.choice(cuisines),
            "restaurant_type": random.choice(restaurant_types),
            "base_prep_time": np.random.uniform(8, 20),
            "capacity_15min": np.random.randint(5, 20),
            "variability_factor": np.random.uniform(0.8, 3.0),
            "rating": round(np.random.uniform(3.5, 4.8), 2)
        })

    return pd.DataFrame(restaurants)


restaurants_df = generate_restaurants(NUM_RESTAURANTS)

# ==============================
# STEP 2: TIME MULTIPLIER
# ==============================

def time_multiplier(hour):
    if 12 <= hour <= 14:
        return 1.8
    elif 19 <= hour <= 22:
        return 2.2
    elif 15 <= hour <= 17:
        return 0.7
    elif 23 <= hour or hour <= 5:
        return 0.5
    else:
        return 1.0


# ==============================
# STEP 3: FAST ORDER GENERATION
# ==============================

all_orders = []
order_id = 0

restaurant_ids = restaurants_df["restaurant_id"].values
base_prep_times = restaurants_df["base_prep_time"].values
capacities = restaurants_df["capacity_15min"].values
variabilities = restaurants_df["variability_factor"].values
ratings = restaurants_df["rating"].values

for day in range(DAYS):

    for minute_block in range(0, 24 * 60, TIME_INTERVAL_MIN):

        current_time = START_DATE + timedelta(days=day, minutes=minute_block)
        hour = current_time.hour
        weekend = current_time.weekday() >= 5

        tm = time_multiplier(hour)
        if weekend:
            tm *= 1.3

        rush_multiplier = 1
        if np.random.rand() < 0.05:
            rush_multiplier = np.random.uniform(1.5, 2.5)

        base_demand = 1.2
        arrival_rate = base_demand * tm * rush_multiplier

        # Vectorized Poisson sampling
        num_orders_array = np.random.poisson(arrival_rate, size=NUM_RESTAURANTS)

        for i in range(NUM_RESTAURANTS):

            num_orders = num_orders_array[i]
            if num_orders == 0:
                continue

            for _ in range(num_orders):

                all_orders.append({
                    "order_id": order_id,
                    "restaurant_id": restaurant_ids[i],
                    "order_time": current_time,
                    "hour": hour,
                    "weekend": weekend,
                    "num_items": np.random.choice([1,2,3,4,5,6], p=[0.35,0.25,0.15,0.1,0.1,0.05]),
                    "customization": np.random.choice([0,1,2,3], p=[0.6,0.25,0.1,0.05]),
                    "order_value": np.random.uniform(150, 1200),
                    "base_prep_time": base_prep_times[i],
                    "capacity_15min": capacities[i],
                    "variability_factor": variabilities[i],
                    "rating": ratings[i]
                })

                order_id += 1

orders_df = pd.DataFrame(all_orders)

# ==============================
# STEP 4: FAST QUEUE SIMULATION
# ==============================

orders_df = orders_df.sort_values(["restaurant_id", "order_time"]).reset_index(drop=True)

active_orders_list = []
load_factor_list = []
kpt_list = []

current_restaurant = None
active_heap = []

for idx, row in orders_df.iterrows():

    if row["restaurant_id"] != current_restaurant:
        active_heap = []
        current_restaurant = row["restaurant_id"]

    current_time = row["order_time"]

    # Remove completed orders efficiently
    while active_heap and active_heap[0] <= current_time:
        heapq.heappop(active_heap)

    active_count = len(active_heap)

    capacity = row["capacity_15min"]
    load_factor = max(0, active_count - capacity / 3)

    peak_bonus = 3 if (12 <= row["hour"] <= 14 or 19 <= row["hour"] <= 22) else 0
    congestion_penalty = 1.2 * (load_factor ** 1.3)

    extreme_delay = np.random.uniform(5, 15) if np.random.rand() < 0.03 else 0
    noise = np.random.normal(0, row["variability_factor"])

    kpt = (
        row["base_prep_time"]
        + 0.9 * row["num_items"]
        + 0.6 * row["customization"]
        + congestion_penalty
        + peak_bonus
        + extreme_delay
        + noise
    )

    kpt = max(5, kpt)

    completion_time = current_time + timedelta(minutes=kpt)
    heapq.heappush(active_heap, completion_time)

    active_orders_list.append(active_count)
    load_factor_list.append(load_factor)
    kpt_list.append(kpt)

orders_df["active_orders"] = active_orders_list
orders_df["load_factor"] = load_factor_list
orders_df["KPT"] = np.round(kpt_list, 2)

# ==============================
# STEP 5: SUMMARY
# ==============================

print("Total Orders Generated:", len(orders_df))
print("Average KPT:", round(orders_df["KPT"].mean(), 2))
print("90th Percentile KPT:", round(np.percentile(orders_df["KPT"], 90), 2))

orders_df.to_csv("synthetic_kpt_dataset.csv", index=False)

print("Dataset saved as synthetic_kpt_dataset.csv")

Total Orders Generated: 155056
Average KPT: 2162.71
90th Percentile KPT: 7706.2
Dataset saved as synthetic_kpt_dataset.csv


In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import heapq
import random

np.random.seed(42)
random.seed(42)

# ==========================================
# CONFIGURATION
# ==========================================

N_RESTAURANTS = 200
DAYS = 30
TIME_SLOT_MIN = 5
START_DATE = datetime(2024, 1, 1)

# ==========================================
# CUISINE CONFIG
# ==========================================

cuisine_config = {
    "Fast Food": (8, 15),
    "Cafe": (10, 18),
    "North Indian": (18, 30),
    "South Indian": (12, 22),
    "Chinese": (15, 25),
    "Biryani": (20, 40),
    "Pizza": (15, 25),
    "Continental": (25, 45),
    "Desserts": (8, 18)
}

weather_types = ["Sunny", "Cloudy", "Rainy", "Stormy"]

weather_demand_factor = {
    "Sunny": 1.0,
    "Cloudy": 1.05,
    "Rainy": 1.2,
    "Stormy": 1.35
}

# ==========================================
# RESTAURANT CREATION
# ==========================================

restaurants = []

for i in range(N_RESTAURANTS):

    cuisine = random.choice(list(cuisine_config.keys()))
    base_low, base_high = cuisine_config[cuisine]

    rating = np.round(np.random.uniform(3.0, 5.0), 1)
    chef_count = np.random.randint(4, 20)
    productivity = np.random.uniform(2.0, 3.5)

    capacity = int(chef_count * productivity)
    base_prep = np.random.randint(base_low, base_high)

    efficiency = np.clip(
        0.7 + (rating - 3.0) * 0.25 + np.random.normal(0, 0.05),
        0.7, 1.3
    )

    popularity_score = rating * np.random.uniform(0.8, 1.2)

    restaurants.append({
        "restaurant_id": i,
        "cuisine_type": cuisine,
        "base_prep_time": base_prep,
        "restaurant_capacity": capacity,
        "restaurant_rating": rating,
        "efficiency_score": efficiency,
        "chef_count": chef_count,
        "popularity_score": popularity_score
    })

restaurants_df = pd.DataFrame(restaurants)

# Normalize popularity for weighted sampling
popularity_probs = restaurants_df["popularity_score"]
popularity_probs = popularity_probs / popularity_probs.sum()

# ==========================================
# TIME DEMAND MULTIPLIER
# ==========================================

def time_demand_multiplier(hour):
    if 12 <= hour <= 14:
        return 1.8
    elif 19 <= hour <= 22:
        return 2.2
    elif 15 <= hour <= 17:
        return 0.7
    elif hour >= 23 or hour <= 5:
        return 0.5
    else:
        return 1.0

# ==========================================
# SIMULATION STATE
# ==========================================

active_heaps = {i: [] for i in range(N_RESTAURANTS)}
data = []

# ==========================================
# MAIN SIMULATION LOOP
# ==========================================

for day in range(DAYS):

    for minute_block in range(0, 24 * 60, TIME_SLOT_MIN):

        current_time = START_DATE + timedelta(days=day, minutes=minute_block)
        hour = current_time.hour
        weekend = current_time.weekday() >= 5

        # Global environment
        weather = np.random.choice(weather_types, p=[0.5,0.2,0.2,0.1])
        festival_flag = np.random.choice([0,1], p=[0.9,0.1])

        demand_multiplier = time_demand_multiplier(hour)

        if weekend:
            demand_multiplier *= 1.3

        demand_multiplier *= weather_demand_factor[weather]

        if festival_flag:
            demand_multiplier *= 1.5

        base_arrival_rate = 0.8

        total_orders_this_slot = np.random.poisson(
            base_arrival_rate * demand_multiplier * N_RESTAURANTS
        )

        if total_orders_this_slot == 0:
            continue

        # Weighted restaurant selection
        chosen_restaurants = np.random.choice(
            restaurants_df["restaurant_id"],
            size=total_orders_this_slot,
            p=popularity_probs
        )

        for rest_id in chosen_restaurants:

            rest = restaurants_df.iloc[rest_id]

            heap = active_heaps[rest_id]

            # Remove completed orders
            while heap and heap[0] <= current_time:
                heapq.heappop(heap)

            active_orders = len(heap)

            # Order features
            total_items = np.random.randint(1, 8)
            item_complexity = total_items * np.random.uniform(1.0, 3.0)

            kitchen_util = active_orders / rest.restaurant_capacity

            queue_length = max(active_orders - rest.chef_count, 0)

            # Prep calculation (nonlinear congestion)
            prep = rest.base_prep_time

            prep += item_complexity * 1.4

            prep += (queue_length ** 1.2) * 0.8

            prep += kitchen_util * 12

            prep -= rest.efficiency_score * 5

            # Weather slows prep
            if weather in ["Rainy","Stormy"]:
                prep *= 1.05

            # Rare heavy delay (heavy tail)
            if np.random.rand() < 0.02:
                prep *= np.random.uniform(1.5, 2.5)

            prep += np.random.normal(0, 2)

            prep = max(5, prep)

            completion_time = current_time + timedelta(minutes=prep)
            heapq.heappush(heap, completion_time)

            data.append({
                "order_time": current_time,
                "restaurant_id": rest_id,
                "cuisine_type": rest.cuisine_type,
                "restaurant_capacity": rest.restaurant_capacity,
                "restaurant_rating": rest.restaurant_rating,
                "chef_count": rest.chef_count,
                "efficiency_score": rest.efficiency_score,
                "total_items": total_items,
                "item_complexity_score": round(item_complexity,2),
                "active_orders_in_kitchen": active_orders,
                "queue_length": queue_length,
                "kitchen_utilization": round(kitchen_util,2),
                "weather": weather,
                "festival_flag": festival_flag,
                "hour": hour,
                "weekend": int(weekend),
                "prep_time": round(prep,2)
            })

# ==========================================
# FINAL DATAFRAME
# ==========================================

df = pd.DataFrame(data)

print("Total Orders:", len(df))
print("Average Prep Time:", round(df["prep_time"].mean(),2))
print("90th Percentile:", round(np.percentile(df["prep_time"],90),2))

df.to_csv("zomato_kpt_realistic_advanced.csv", index=False)

print("Dataset generated successfully.")

Total Orders: 1904100
Average Prep Time: 64.31
90th Percentile: 80.59
Dataset generated successfully.


In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import heapq
import random

np.random.seed(42)
random.seed(42)

# ==========================================
# CONFIGURATION (CALIBRATED)
# ==========================================

N_RESTAURANTS = 200
DAYS = 30
TIME_SLOT_MIN = 5
START_DATE = datetime(2024, 1, 1)

# Lowered arrival rate for stability
BASE_ARRIVAL_RATE = 0.18

# ==========================================
# CUISINE CONFIG (CALIBRATED RANGES)
# ==========================================

cuisine_config = {
    "Fast Food": (8, 14),
    "Cafe": (10, 16),
    "North Indian": (15, 25),
    "South Indian": (12, 20),
    "Chinese": (14, 22),
    "Biryani": (18, 32),
    "Pizza": (14, 22),
    "Continental": (20, 35),
    "Desserts": (8, 16)
}

weather_types = ["Sunny", "Cloudy", "Rainy", "Stormy"]

# Reduced storm frequency & impact
weather_demand_factor = {
    "Sunny": 1.0,
    "Cloudy": 1.05,
    "Rainy": 1.15,
    "Stormy": 1.25
}

weather_probs = [0.6, 0.25, 0.12, 0.03]

# ==========================================
# RESTAURANT CREATION (WITH POPULARITY SKEW)
# ==========================================

restaurants = []

for i in range(N_RESTAURANTS):

    cuisine = random.choice(list(cuisine_config.keys()))
    base_low, base_high = cuisine_config[cuisine]

    rating = np.round(np.random.uniform(3.0, 5.0), 1)
    chef_count = np.random.randint(4, 18)
    productivity = np.random.uniform(2.0, 3.0)

    capacity = int(chef_count * productivity)
    base_prep = np.random.randint(base_low, base_high)

    efficiency = np.clip(
        0.75 + (rating - 3.0) * 0.2 + np.random.normal(0, 0.04),
        0.75, 1.25
    )

    # Power-law skew (stronger than uniform)
    popularity_score = (rating ** 3) * np.random.uniform(0.7, 1.3)

    restaurants.append({
        "restaurant_id": i,
        "cuisine_type": cuisine,
        "base_prep_time": base_prep,
        "restaurant_capacity": capacity,
        "restaurant_rating": rating,
        "efficiency_score": efficiency,
        "chef_count": chef_count,
        "popularity_score": popularity_score
    })

restaurants_df = pd.DataFrame(restaurants)

# Normalize popularity probabilities
popularity_probs = restaurants_df["popularity_score"]
popularity_probs = popularity_probs / popularity_probs.sum()

# ==========================================
# TIME DEMAND MULTIPLIER
# ==========================================

def time_demand_multiplier(hour):
    if 12 <= hour <= 14:
        return 1.7
    elif 19 <= hour <= 22:
        return 2.0
    elif 15 <= hour <= 17:
        return 0.7
    elif hour >= 23 or hour <= 5:
        return 0.5
    else:
        return 1.0

# ==========================================
# SIMULATION STATE
# ==========================================

active_heaps = {i: [] for i in range(N_RESTAURANTS)}
data = []

# ==========================================
# MAIN SIMULATION LOOP
# ==========================================

for day in range(DAYS):

    for minute_block in range(0, 24 * 60, TIME_SLOT_MIN):

        current_time = START_DATE + timedelta(days=day, minutes=minute_block)
        hour = current_time.hour
        weekend = current_time.weekday() >= 5

        weather = np.random.choice(weather_types, p=weather_probs)
        festival_flag = np.random.choice([0,1], p=[0.92,0.08])

        demand_multiplier = time_demand_multiplier(hour)

        if weekend:
            demand_multiplier *= 1.25

        demand_multiplier *= weather_demand_factor[weather]

        if festival_flag:
            demand_multiplier *= 1.4

        total_orders_this_slot = np.random.poisson(
            BASE_ARRIVAL_RATE * demand_multiplier * N_RESTAURANTS
        )

        if total_orders_this_slot == 0:
            continue

        chosen_restaurants = np.random.choice(
            restaurants_df["restaurant_id"],
            size=total_orders_this_slot,
            p=popularity_probs
        )

        for rest_id in chosen_restaurants:

            rest = restaurants_df.iloc[rest_id]
            heap = active_heaps[rest_id]

            # Remove completed orders
            while heap and heap[0] <= current_time:
                heapq.heappop(heap)

            active_orders = len(heap)

            # Order complexity
            total_items = np.random.randint(1, 7)
            item_complexity = total_items * np.random.uniform(1.0, 2.5)

            kitchen_util = active_orders / rest.restaurant_capacity
            queue_length = max(active_orders - rest.chef_count, 0)

            # ===== PREP CALCULATION (CALIBRATED) =====

            prep = rest.base_prep_time

            prep += item_complexity * 1.2

            # Softer nonlinear congestion
            prep += (queue_length ** 1.1) * 0.5

            prep += kitchen_util * 6

            prep -= rest.efficiency_score * 4

            if weather in ["Rainy","Stormy"]:
                prep *= 1.04

            # Rare heavy tail delay
            if np.random.rand() < 0.015:
                prep *= np.random.uniform(1.4, 2.0)

            prep += np.random.normal(0, 1.8)

            prep = max(5, prep)

            completion_time = current_time + timedelta(minutes=prep)
            heapq.heappush(heap, completion_time)

            data.append({
                "order_time": current_time,
                "restaurant_id": rest_id,
                "cuisine_type": rest.cuisine_type,
                "restaurant_capacity": rest.restaurant_capacity,
                "restaurant_rating": rest.restaurant_rating,
                "chef_count": rest.chef_count,
                "efficiency_score": rest.efficiency_score,
                "total_items": total_items,
                "item_complexity_score": round(item_complexity,2),
                "active_orders_in_kitchen": active_orders,
                "queue_length": queue_length,
                "kitchen_utilization": round(kitchen_util,2),
                "weather": weather,
                "festival_flag": festival_flag,
                "hour": hour,
                "weekend": int(weekend),
                "prep_time": round(prep,2)
            })

# ==========================================
# FINAL DATAFRAME
# ==========================================

df = pd.DataFrame(data)

print("Total Orders:", len(df))
print("Average Prep Time:", round(df["prep_time"].mean(),2))
print("Median Prep Time:", round(df["prep_time"].median(),2))
print("90th Percentile:", round(np.percentile(df["prep_time"],90),2))
print("99th Percentile:", round(np.percentile(df["prep_time"],99),2))

df.to_csv("zomato_kpt_realistic_calibrated.csv", index=False)

print("Dataset generated successfully.")

Total Orders: 380463
Average Prep Time: 20.39
Median Prep Time: 19.59
90th Percentile: 30.44
99th Percentile: 40.38
Dataset generated successfully.


In [3]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import heapq
import random

np.random.seed(42)
random.seed(42)

# ==========================================
# CONFIGURATION (CONTROLLED AGGRESSIVE)
# ==========================================

N_RESTAURANTS = 200
DAYS = 30
TIME_SLOT_MIN = 5
START_DATE = datetime(2024, 1, 1)

BASE_ARRIVAL_RATE = 0.22  # Increased for controlled congestion

# ==========================================
# CUISINE CONFIG (REALISTIC RANGES)
# ==========================================

cuisine_config = {
    "Fast Food": (8, 14),
    "Cafe": (10, 16),
    "North Indian": (15, 25),
    "South Indian": (12, 20),
    "Chinese": (14, 22),
    "Biryani": (18, 32),
    "Pizza": (14, 22),
    "Continental": (20, 35),
    "Desserts": (8, 16)
}

weather_types = ["Sunny", "Cloudy", "Rainy", "Stormy"]

weather_demand_factor = {
    "Sunny": 1.0,
    "Cloudy": 1.05,
    "Rainy": 1.15,
    "Stormy": 1.25
}

weather_probs = [0.6, 0.25, 0.12, 0.03]

# ==========================================
# RESTAURANT CREATION (URBAN STRESS)
# ==========================================

restaurants = []

for i in range(N_RESTAURANTS):

    cuisine = random.choice(list(cuisine_config.keys()))
    base_low, base_high = cuisine_config[cuisine]

    rating = np.round(np.random.uniform(3.0, 5.0), 1)

    # Reduced chef count for congestion
    chef_count = np.random.randint(3, 10)

    productivity = np.random.uniform(2.0, 3.0)
    capacity = int(chef_count * productivity)

    base_prep = np.random.randint(base_low, base_high)

    efficiency = np.clip(
        0.75 + (rating - 3.0) * 0.2 + np.random.normal(0, 0.04),
        0.75, 1.25
    )

    popularity_score = (rating ** 3) * np.random.uniform(0.7, 1.3)

    restaurants.append({
        "restaurant_id": i,
        "cuisine_type": cuisine,
        "base_prep_time": base_prep,
        "restaurant_capacity": capacity,
        "restaurant_rating": rating,
        "efficiency_score": efficiency,
        "chef_count": chef_count,
        "popularity_score": popularity_score
    })

restaurants_df = pd.DataFrame(restaurants)

popularity_probs = restaurants_df["popularity_score"]
popularity_probs = popularity_probs / popularity_probs.sum()

# ==========================================
# TIME DEMAND MULTIPLIER
# ==========================================

def time_demand_multiplier(hour):
    if 12 <= hour <= 14:
        return 1.8
    elif 19 <= hour <= 22:
        return 2.2
    elif 15 <= hour <= 17:
        return 0.7
    elif hour >= 23 or hour <= 5:
        return 0.5
    else:
        return 1.0

# ==========================================
# SIMULATION STATE
# ==========================================

active_heaps = {i: [] for i in range(N_RESTAURANTS)}
data = []

# ==========================================
# MAIN SIMULATION LOOP
# ==========================================

for day in range(DAYS):

    for minute_block in range(0, 24 * 60, TIME_SLOT_MIN):

        current_time = START_DATE + timedelta(days=day, minutes=minute_block)
        hour = current_time.hour
        weekend = current_time.weekday() >= 5

        weather = np.random.choice(weather_types, p=weather_probs)
        festival_flag = np.random.choice([0,1], p=[0.92,0.08])

        demand_multiplier = time_demand_multiplier(hour)

        if weekend:
            demand_multiplier *= 1.25

        demand_multiplier *= weather_demand_factor[weather]

        if festival_flag:
            demand_multiplier *= 1.4

        total_orders_this_slot = np.random.poisson(
            BASE_ARRIVAL_RATE * demand_multiplier * N_RESTAURANTS
        )

        if total_orders_this_slot == 0:
            continue

        chosen_restaurants = np.random.choice(
            restaurants_df["restaurant_id"],
            size=total_orders_this_slot,
            p=popularity_probs
        )

        for rest_id in chosen_restaurants:

            rest = restaurants_df.iloc[rest_id]
            heap = active_heaps[rest_id]

            while heap and heap[0] <= current_time:
                heapq.heappop(heap)

            active_orders = len(heap)

            total_items = np.random.randint(1, 7)
            item_complexity = total_items * np.random.uniform(1.0, 2.5)

            kitchen_util = active_orders / rest.restaurant_capacity

            # Congestion begins at 70% utilization
            queue_length = max(
                active_orders - 0.7 * rest.restaurant_capacity,
                0
            )

            # ==============================
            # PREP CALCULATION (AGGRESSIVE BUT CONTROLLED)
            # ==============================

            prep = rest.base_prep_time

            prep += item_complexity * 1.3

            # Stronger nonlinear congestion
            prep += (queue_length ** 1.15) * 0.7

            prep += kitchen_util * 8

            prep -= rest.efficiency_score * 4

            if weather in ["Rainy","Stormy"]:
                prep *= 1.05

            # Heavy tail rare delay
            if np.random.rand() < 0.02:
                prep *= np.random.uniform(1.5, 2.2)

            prep += np.random.normal(0, 2)

            prep = max(5, prep)

            completion_time = current_time + timedelta(minutes=prep)
            heapq.heappush(heap, completion_time)

            data.append({
                "order_time": current_time,
                "restaurant_id": rest_id,
                "cuisine_type": rest.cuisine_type,
                "restaurant_capacity": rest.restaurant_capacity,
                "restaurant_rating": rest.restaurant_rating,
                "chef_count": rest.chef_count,
                "efficiency_score": rest.efficiency_score,
                "total_items": total_items,
                "item_complexity_score": round(item_complexity,2),
                "active_orders_in_kitchen": active_orders,
                "queue_length": round(queue_length,2),
                "kitchen_utilization": round(kitchen_util,2),
                "weather": weather,
                "festival_flag": festival_flag,
                "hour": hour,
                "weekend": int(weekend),
                "prep_time": round(prep,2)
            })

# ==========================================
# FINAL DATAFRAME
# ==========================================

df = pd.DataFrame(data)

print("Total Orders:", len(df))
print("Average Prep Time:", round(df["prep_time"].mean(),2))
print("Median Prep Time:", round(df["prep_time"].median(),2))
print("90th Percentile:", round(np.percentile(df["prep_time"],90),2))
print("99th Percentile:", round(np.percentile(df["prep_time"],99),2))
print("Mean Queue Length:", round(df["queue_length"].mean(),2))

df.to_csv("zomato_kpt__final_dataset.csv", index=False)

print("Dataset generated successfully.")

Total Orders: 484692
Average Prep Time: 22.35
Median Prep Time: 21.19
90th Percentile: 33.52
99th Percentile: 48.7
Mean Queue Length: 0.08
Dataset generated successfully.


In [4]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import heapq
import random

np.random.seed(42)
random.seed(42)

# ==========================================
# CONFIGURATION (CONTROLLED AGGRESSIVE)
# ==========================================

N_RESTAURANTS = 200
DAYS = 30
TIME_SLOT_MIN = 5
START_DATE = datetime(2024, 1, 1)

BASE_ARRIVAL_RATE = 0.25  # Increased for controlled congestion

# ==========================================
# CUISINE CONFIG (REALISTIC RANGES)
# ==========================================

cuisine_config = {
    "Fast Food": (8, 14),
    "Cafe": (10, 16),
    "North Indian": (15, 25),
    "South Indian": (12, 20),
    "Chinese": (14, 22),
    "Biryani": (18, 32),
    "Pizza": (14, 22),
    "Continental": (20, 35),
    "Desserts": (8, 16)
}

weather_types = ["Sunny", "Cloudy", "Rainy", "Stormy"]

weather_demand_factor = {
    "Sunny": 1.0,
    "Cloudy": 1.05,
    "Rainy": 1.15,
    "Stormy": 1.25
}

weather_probs = [0.6, 0.25, 0.12, 0.03]

# ==========================================
# RESTAURANT CREATION (URBAN STRESS)
# ==========================================

restaurants = []

for i in range(N_RESTAURANTS):

    cuisine = random.choice(list(cuisine_config.keys()))
    base_low, base_high = cuisine_config[cuisine]

    rating = np.round(np.random.uniform(3.0, 5.0), 1)

    # Reduced chef count for congestion
    chef_count = np.random.randint(3, 10)

    productivity = np.random.uniform(1.6, 2.2)
    capacity = int(chef_count * productivity)

    base_prep = np.random.randint(base_low, base_high)

    efficiency = np.clip(
        0.75 + (rating - 3.0) * 0.2 + np.random.normal(0, 0.04),
        0.75, 1.25
    )

    popularity_score = (rating ** 3) * np.random.uniform(0.7, 1.3)

    restaurants.append({
        "restaurant_id": i,
        "cuisine_type": cuisine,
        "base_prep_time": base_prep,
        "restaurant_capacity": capacity,
        "restaurant_rating": rating,
        "efficiency_score": efficiency,
        "chef_count": chef_count,
        "popularity_score": popularity_score
    })

restaurants_df = pd.DataFrame(restaurants)

popularity_probs = restaurants_df["popularity_score"]
popularity_probs = popularity_probs / popularity_probs.sum()

# ==========================================
# TIME DEMAND MULTIPLIER
# ==========================================

def time_demand_multiplier(hour):
    if 12 <= hour <= 14:
        return 1.8
    elif 19 <= hour <= 22:
        return 2.2
    elif 15 <= hour <= 17:
        return 0.7
    elif hour >= 23 or hour <= 5:
        return 0.5
    else:
        return 1.0

# ==========================================
# SIMULATION STATE
# ==========================================

active_heaps = {i: [] for i in range(N_RESTAURANTS)}
data = []

# ==========================================
# MAIN SIMULATION LOOP
# ==========================================

for day in range(DAYS):

    for minute_block in range(0, 24 * 60, TIME_SLOT_MIN):

        current_time = START_DATE + timedelta(days=day, minutes=minute_block)
        hour = current_time.hour
        weekend = current_time.weekday() >= 5

        weather = np.random.choice(weather_types, p=weather_probs)
        festival_flag = np.random.choice([0,1], p=[0.92,0.08])

        demand_multiplier = time_demand_multiplier(hour)

        if weekend:
            demand_multiplier *= 1.25

        demand_multiplier *= weather_demand_factor[weather]

        if festival_flag:
            demand_multiplier *= 1.4

        total_orders_this_slot = np.random.poisson(
            BASE_ARRIVAL_RATE * demand_multiplier * N_RESTAURANTS
        )

        if total_orders_this_slot == 0:
            continue

        chosen_restaurants = np.random.choice(
            restaurants_df["restaurant_id"],
            size=total_orders_this_slot,
            p=popularity_probs
        )

        for rest_id in chosen_restaurants:

            rest = restaurants_df.iloc[rest_id]
            heap = active_heaps[rest_id]

            while heap and heap[0] <= current_time:
                heapq.heappop(heap)

            active_orders = len(heap)

            total_items = np.random.randint(1, 7)
            item_complexity = total_items * np.random.uniform(1.0, 2.5)

            kitchen_util = active_orders / rest.restaurant_capacity

            # Congestion begins at 70% utilization
            queue_length = max(
                active_orders - 0.6 * rest.restaurant_capacity,
                0
            )

            # ==============================
            # PREP CALCULATION (AGGRESSIVE BUT CONTROLLED)
            # ==============================

            prep = rest.base_prep_time

            prep += item_complexity * 1.3

            # Stronger nonlinear congestion
            prep += (queue_length ** 1.15) * 0.7

            prep += kitchen_util * 8

            prep -= rest.efficiency_score * 4

            if weather in ["Rainy","Stormy"]:
                prep *= 1.05

            # Heavy tail rare delay
            if np.random.rand() < 0.02:
                prep *= np.random.uniform(1.5, 2.2)

            prep += np.random.normal(0, 2)

            prep = max(5, prep)

            completion_time = current_time + timedelta(minutes=prep)
            heapq.heappush(heap, completion_time)

            data.append({
                "order_time": current_time,
                "restaurant_id": rest_id,
                "cuisine_type": rest.cuisine_type,
                "restaurant_capacity": rest.restaurant_capacity,
                "restaurant_rating": rest.restaurant_rating,
                "chef_count": rest.chef_count,
                "efficiency_score": rest.efficiency_score,
                "total_items": total_items,
                "item_complexity_score": round(item_complexity,2),
                "active_orders_in_kitchen": active_orders,
                "queue_length": round(queue_length,2),
                "kitchen_utilization": round(kitchen_util,2),
                "weather": weather,
                "festival_flag": festival_flag,
                "hour": hour,
                "weekend": int(weekend),
                "prep_time": round(prep,2)
            })

# ==========================================
# FINAL DATAFRAME
# ==========================================

df = pd.DataFrame(data)

print("Total Orders:", len(df))
print("Average Prep Time:", round(df["prep_time"].mean(),2))
print("Median Prep Time:", round(df["prep_time"].median(),2))
print("90th Percentile:", round(np.percentile(df["prep_time"],90),2))
print("99th Percentile:", round(np.percentile(df["prep_time"],99),2))
print("Mean Queue Length:", round(df["queue_length"].mean(),2))

df.to_csv("zomato_kpt__final_dataset.csv", index=False)

print("Dataset generated successfully.")

Total Orders: 550502
Average Prep Time: 23.63
Median Prep Time: 22.06
90th Percentile: 35.79
99th Percentile: 57.57
Mean Queue Length: 0.38
Dataset generated successfully.
